### Bidirektionale Breitensuche beim 2x2x2 Rubiks-Cube

Der 2x2x2 Rubiks-Cube hat total 3'674'160 Zustände.
Jeder Zustand ist im max. 11 Drehungen zu erreichen.
Suchen wir die Lösung zu einem Zustand, genügt es also, von diesen Zustand und vom initialen Zustand
6 Drehungen tief zu suchen.
Mit maximal 6 Drehungen lassen sich nur 62360 erreichen.
Statt max. 3'674'160, brauchen wir nur ca. 2 x  62'360 Zustände zu besuchen.
Hier lohnt sich bidirektionale Breitensuche.


**Würfel-Zustände und Operationen**  
Wir platzieren den  2x2x2 Rubiks-Cube (Würfel) so, dass weiss oben und rot vorne ist
(damit ist gelb unten und orange hinten).  
Die Hauptfarbe eines Würfelchens ist weiss oder gelb.
Ein  Würfelchen mit Hauptfarbe weiss oder gelb hat Orientierung 0.  
Rotiert man ein Würfelchen mit Orientierung 0 um 1x$120^{\circ}$ (2x$120^{\circ}$) im Uhrzeigersinn um die Achse
vom Würfelzentrum zur Ecke des Würfelchens, dann hat das Würfelchen Orientierung 1 (bez. 2).



Wir drehen jeweils nur das obere Stockwerk, die rechte Seite und die Vorderseite. 
Drehungen im Uhrzeigersinn werden mit U(p), R(ight), (F)ront) bezeichnet,
drehungen im Gegenuhrzeigersinn mit u, r, f und Doppeldrehungen mit
U2, R2 und F2.


Der Zustand des Würfels wird mit 2 Tupeln der Länge 8 beschreiben.
- Das 1. Tuple ist eine Permutation der Zahlen 0 bis 7.  
  `(3, 0, 1, 2, 4, 5, 6, 7)` ist so zu lesen:  
  Würfelchen 3 ist auf Position 0 (dort wo Würfelchen 0 im initialen Zustand ist), Würfelchen 0 ist auf Position 1, u.s.w.  
  Das oberste Stockwerk hat sich also im Uhrzeigersinn gedreht, 
  die Würfelchen des unteren Stockwerks bleiben auf ihren Positionen.
- Das 2. Tuple enthält die Zahlen 0,1 und 2.  
  `(1, 2, 2, 1, 0, 0, 0, 0)` gibt die Orientierung der Würfelchen an.
  
Zum Beispiel ist nach Drehen von R der Würfel im Zustand  
`(0, 2, 5, 3, 4, 6, 1, 7) + (0, 1, 2, 0, 0, 1, 2, 0)`.  


Jeder Zustand kann auch als Operation aufgefasst werden, nämlich als die Operation, welche den
initialen Zustand in den gegebenen überführt.

Das Modul `babycube` stellt eine Funktion `apply_op(op, state=ID)` bereit,
die eine Operation `op` aus ('F', 'R', 'U', 'F2', 'f', 'R2', 'r', 'U2', 'u') auf den Zustand `state` anwendet und den resultiereden Zustand zurück gibt. 

In [26]:
import babycube as cube

cube.ID

(0, 1, 2, 3, 4, 5, 6, 7, 0, 0, 0, 0, 0, 0, 0, 0)

In [27]:
cube.inv_key('R')

'r'

In [28]:
s = cube.apply_op('R')
s

(0, 2, 5, 3, 4, 6, 1, 7, 0, 1, 2, 0, 0, 1, 2, 0)

In [29]:
cube.apply_op('r', s)

(0, 1, 2, 3, 4, 5, 6, 7, 0, 0, 0, 0, 0, 0, 0, 0)

***
Eine Folge von Drehungen (Operationen) geben wir als
Wort der Form `'RF2R2u'` an. Um diese Operationen rückgänging zu machen,
ist `'UR2F2r`' anzuwenden.
***

In [31]:
def get_ops(word):
    '''gib die Operationen aus dem Wort zurueck'''
    n = len(word)
    i = 0
    while i < n-1:
        j = i + 1 + (word[i+1] == '2')
        yield word[i:j]
        i = j

    if i < n:
        yield word[i]


def apply_ops(ops, s=cube.ID):
    '''Fuehre eine Liste/Tuple von Operationen aus'''
    for k in ops:
        s = cube.apply_op(k, s)
    return s


def apply_word(word, s=cube.ID):
    '''fuehre ein Wort aus'''
    for k in get_ops(word):
        s = cube.apply_op(k, s)
    return s


def inv_word(word):
    '''liefert Wort, welche umkekehrte Operation ausfuehrt'''
    keys = list(get_ops(word))
    return ''.join(cube.inv_key(k) for k in reversed(keys))

In [32]:
word = 'RUFU2rf'
ops = list(get_ops(word))
ops

['R', 'U', 'F', 'U2', 'r', 'f']

In [33]:
state1 = apply_ops(ops)
state2 = apply_word(word)
state1 == state2, state1

(True, (5, 1, 3, 4, 0, 6, 2, 7, 0, 0, 2, 0, 1, 1, 2, 0))

In [34]:
word = 'RUF2U2rUrUR2fU2FurfU2'
s = apply_word(word)
s

(4, 1, 3, 5, 0, 2, 6, 7, 1, 0, 2, 0, 2, 0, 1, 0)

In [35]:
w = inv_word(word)
w

'U2FRUfU2FR2uRuRU2F2ur'

In [36]:
apply_word(w, s)

(0, 1, 2, 3, 4, 5, 6, 7, 0, 0, 0, 0, 0, 0, 0, 0)

***
Wir nutzen Breitensuche, um alle Zustände die in n Drehungen erreichbar sind zu finden.
Dazu speichern wir im Deque `nodes_to_visit` jeweils Tuple der Form `(depth, state)`.  
**Beachte**: Sobald der letzte Knoten mit Tiefe $n$ von rechts her aus dem Deck entfernt wurde,
befinden sich nur noch Knoten der Tiefe $n+1$ im Deque, und war alle Knoten dieser Tiefe.
***

In [69]:
from collections import deque


def get_neighbors(state):
    '''gibt Name der Operation und den neues Zustand zurueck'''
    for op in ('F', 'R', 'U', 'F2', 'f', 'R2', 'r', 'U2', 'u'):
        new_state = cube.apply_op(op, state)
        yield op, new_state


def search_bf(state, get_options, max_depth=11):
    depth = 0
    nodes_to_visit = deque([(depth, state)])
    go_back = {state: None}

    while nodes_to_visit:
        depth, state = nodes_to_visit.pop()
        if depth == max_depth:
            return state, go_back

        for op, new_state in get_options(state):
            if new_state in go_back:
                continue
            go_back[new_state] = cube.inv_key(op), state
            nodes_to_visit.appendleft((depth+1, new_state))


def get_path_home(state, go_back):
    path = []
    while (dnode := go_back[state]):
        op, state = dnode
        path.append(op)
    return path


def inv_path(path):
    return [cube.inv_key(k) for k in reversed(path)]

In [74]:
state, go_back = search_bf(cube.ID, get_neighbors, max_depth=6)
state, len(go_back)

((6, 0, 4, 5, 1, 2, 3, 7, 1, 0, 1, 1, 1, 1, 1, 0), 62360)

In [75]:
solution = get_path_home(state, go_back)
solution

['u', 'f', 'r', 'f', 'r', 'f']

In [76]:
apply_ops(solution, state)

(0, 1, 2, 3, 4, 5, 6, 7, 0, 0, 0, 0, 0, 0, 0, 0)

In [78]:
scramble = inv_path(solution)
scramble

['F', 'R', 'F', 'R', 'F', 'U']

In [79]:
scramble = inv_path(solution)
apply_ops(scramble) == state

True

In [77]:
depth_n_states = {}
count = 0
for i in range(7):
    _, go_back = search_bf(cube.ID, get_neighbors, max_depth=i)
    depth_n_states[i] = len(go_back) - count
    count = len(go_back)

depth_n_states

{0: 1, 1: 9, 2: 54, 3: 321, 4: 1847, 5: 9992, 6: 50136}

### Aufgabe
Benutze folgende FUnktion, um den Würfel zu verdrehen.
```python
import random


def scramble_cube(n=100):
    ops = random.choices(list(cube.OPS), k=n)
    return apply_ops(ops)
```
Verwende bidirektionale Breitensuche, um den Weg von einem verdrehten Würfel zum initialen Zustand zu finden.